# Term Sheet Extraction — POC

End-to-end test for the Gemini-based term-sheet extractor.

**What this notebook does:**
1. Lists PDFs available in `data/term_sheets_samples/`.
2. Sends the chosen PDF to Gemini, parses the response into a portfolio row.
3. Runs schema validation on the extracted dict.
4. Prompts the user for **trading-side fields** the term sheet can't know:
   - `cost_price` — what you actually paid (as a fraction of denomination, e.g. `1.00` for at-par)
   - `purchase_date` — when you bought it (YYYY-MM-DD)
   - `position_units` OR `notional` — your holding size
5. Merges everything into a complete row that drops straight into the same schema used by `data/portfolio.py`.

**Prerequisite:** add `GEMINI_API_KEY` to `.streamlit/secrets.toml`.

## 0 — Setup

In [ ]:
# Make project root importable when running from notebooks/
import sys, os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print(f'Working directory: {Path.cwd()}')

In [ ]:
from src.term_sheet_extractor import extract_term_sheet, validate
import json, datetime as dt

## 1 — Pick a PDF

In [ ]:
SAMPLES_DIR = Path('data/term_sheets_samples')
pdfs = sorted(SAMPLES_DIR.glob('*.pdf'))

if not pdfs:
    print(f'No PDFs in {SAMPLES_DIR}/. Drop one in and re-run this cell.')
else:
    print(f'{len(pdfs)} PDF(s) found:')
    for i, p in enumerate(pdfs):
        print(f'  [{i}] {p.name}')

In [ ]:
# Edit the index to pick a different PDF.
PDF_INDEX = 0
pdf_path = pdfs[PDF_INDEX]
print(f'Selected: {pdf_path.name}')

## 2 — Extract via Gemini

Single call. Takes ~3-8 seconds for a typical term sheet.

In [ ]:
row = extract_term_sheet(pdf_path)
print(json.dumps(row, indent=2, ensure_ascii=False))

## 3 — Validate

Flags missing required fields, unknown product type, or list-length mismatches.

In [ ]:
warnings = validate(row)
if warnings:
    print(f'{len(warnings)} warning(s):')
    for w in warnings:
        print(f'  ! {w}')
else:
    print('Validation: clean.')

## 4 — Add trading-side fields

These fields can't be in the term sheet — they're what you actually did when you bought the product. Edit the values below to match your trade.

**Sizing:** provide *either* `position_units` (number of certificates you hold) *or* `notional` (total face value). The other will be derived from `denomination`. If you set both, `notional` wins.

In [ ]:
# ===== EDIT THESE VALUES =====
COST_PRICE     = 1.00            # fraction of denomination — 1.00 = bought at par
PURCHASE_DATE  = '2026-03-04'    # YYYY-MM-DD
POSITION_UNITS = 10              # number of certificates you hold
NOTIONAL       = None            # OR set notional directly; leave None to derive from units
# =============================

denom = row.get('denomination')
if denom is None:
    raise ValueError('No denomination extracted — cannot compute notional. Set NOTIONAL manually.')

if NOTIONAL is not None:
    notional = float(NOTIONAL)
    position_units = int(round(notional / denom))
else:
    position_units = int(POSITION_UNITS)
    notional = float(denom) * position_units

# Sanity-check the date format
dt.datetime.strptime(PURCHASE_DATE, '%Y-%m-%d')

print(f'cost_price     = {COST_PRICE}')
print(f'purchase_date  = {PURCHASE_DATE}')
print(f'position_units = {position_units}')
print(f'notional       = {notional:,.2f}  (denomination {denom} × {position_units} units)')

## 5 — Build the complete portfolio row

Merges the extracted fields with the trading-side ones. The shape matches `data/portfolio.py` rows exactly, so this dict can be dropped straight into the demo portfolio for testing.

In [ ]:
complete_row = {
    **row,
    'cost_price':     COST_PRICE,
    'purchase_date':  PURCHASE_DATE,
    'position_units': position_units,
    'notional':       notional,
    # Live market data — leave as None for now; populated later from prices.csv.
    'current_spots':       [None] * len(row.get('underlyings', []) or []),
    'current_spot_dates':  [None] * len(row.get('underlyings', []) or []),
    'barrier_breached':    False,
}

print(json.dumps(complete_row, indent=2, ensure_ascii=False, default=str))

## 6 — (Optional) Smoke-test the row through the product class

If the extracted product is a CPN, instantiate it and print the summary to check the row actually works end-to-end.

In [ ]:
import pandas as pd

ptype = complete_row.get('product_type', '').upper()
row_series = pd.Series(complete_row)

if ptype == 'CPN':
    from src.capital_protection_note import CapitalProtectionNote
    try:
        prod = CapitalProtectionNote(row_series, final_level=0.0)
        summary = prod.summary()
        print('CPN instantiated. Summary:')
        for k, v in summary.items():
            print(f'  {k:30s} = {v}')
    except Exception as e:
        print(f'CPN construction failed: {type(e).__name__}: {e}')
elif ptype in ('BRC', 'MBRC', 'AC_BRC'):
    from src.reverse_convertible import ReverseConvertible
    try:
        prod = ReverseConvertible(row_series)
        summary = prod.summary()
        print(f'{ptype} instantiated. Summary:')
        for k, v in summary.items():
            print(f'  {k:30s} = {v}')
    except Exception as e:
        print(f'{ptype} construction failed: {type(e).__name__}: {e}')
else:
    print(f'Unknown product_type: {ptype!r} — skipping smoke test.')